# Детектор рассогласования текста и оценки

Ловим отзывы, где текст не соответствует выставленной звезде. Это нужно для двух вещей: чистить рейтинг, понижая вес рассогласованных оценок и помечая их как шумные для рекомендателя, и подсказывать автору прямо при публикации, если текст звучит иначе, чем выбранная оценка.

Реальных меток фейк или ошибка у нас нет, поэтому датасет синтетический: в scripts/build_mismatch_dataset.py половину отзывов оставили как есть с меткой 0, а второй половине испортили оценку и пометили меткой 1. Распределение оценок в обоих классах сделали одинаковым, так что по одной звезде метку не угадать.

Чтобы проверить это честно, мы учим три модели на одних и тех же данных: только по звезде, только по тексту и по тексту вместе со звездой. Берём лишь text и stars, исходную оценку и id не трогаем, иначе это утечка. Метрика - ROC-AUC, синтетические метки позволяют мерить честно.

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import time
import re
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve, precision_recall_fscore_support

_PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(_PROJECT))
from _constants import MISMATCH_PARQUET, ARTIFACTS, REPORTS, ENABLE_ARTIFACTS, ENABLE_LOGGING, ENABLE_FAST_DEV_RUN

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

SMOKE = ENABLE_FAST_DEV_RUN
SAMPLE_TOTAL = 3000 if SMOKE else 200_000
MAXLEN = 64 if SMOKE else 200
EPOCHS = 1 if SMOKE else 5
BS = 128

def savefig(name):
    if not ENABLE_ARTIFACTS:
        return
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    plt.savefig(ARTIFACTS / f"Task2_mismatch_{name}.png", bbox_inches="tight")

device

## 1. Данные

Грузим синтетический датасет, признаки text и stars, таргет label. Делим на train, val и test со стратификацией по метке.

In [ ]:
df = pd.read_parquet(MISMATCH_PARQUET, columns=["text", "stars", "label"]).dropna(subset=["text"])
df = df[df["text"].str.len() > 0].copy()
df["stars"] = df["stars"].astype(int)

if SAMPLE_TOTAL and len(df) > SAMPLE_TOTAL:
    df = df.sample(SAMPLE_TOTAL, random_state=SEED).reset_index(drop=True)

train_df, tmp = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp["label"], random_state=SEED)

len(train_df), len(val_df), len(test_df)

Распределение оценок в двух классах должно совпадать, иначе по звезде была бы лазейка. Проверяем.

In [ ]:
df.groupby("label")["stars"].value_counts(normalize=True).unstack().round(3)

Доли оценок в классах 0 и 1 совпадают, значит по одной звезде метку не угадать.

## 2. Токенизация

Токенайзер и словарь как в соседнем ноутбуке: слова в нижнем регистре, словарь строим по train, редкие слова в unk. Звезду переводим из 1..5 в индекс 0..4 для эмбеддинга.

In [ ]:
TOKEN_RE = re.compile(r"[a-z]+")

def tokenize(t):
    return TOKEN_RE.findall(str(t).lower())

MIN_FREQ = 2
cnt = Counter()
for t in train_df["text"]:
    cnt.update(tokenize(t))
itos = ["<pad>", "<unk>"] + [w for w, c in cnt.items() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB = len(itos)
PAD_IDX = 0
MIN_LEN = 5

def encode(t):
    return [stoi.get(w, 1) for w in tokenize(t)][:MAXLEN]

class MismatchDS(Dataset):
    def __init__(self, d):
        self.x = [encode(t) for t in d["text"]]
        self.s = (d["stars"].to_numpy() - 1).astype(int)
        self.y = d["label"].to_numpy().astype(int)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.s[i], self.y[i]

def collate(batch):
    xs, ss, ys = zip(*batch)
    maxlen = max(MIN_LEN, max(len(x) for x in xs))
    ids = torch.zeros(len(xs), maxlen, dtype=torch.long)
    for i, x in enumerate(xs):
        if x:
            ids[i, :len(x)] = torch.tensor(x, dtype=torch.long)
    return ids, torch.tensor(ss, dtype=torch.long), torch.tensor(ys, dtype=torch.long)

def make_loader(d, shuffle):
    return DataLoader(MismatchDS(d), batch_size=BS, shuffle=shuffle, collate_fn=collate)

tr_loader = make_loader(train_df, True)
va_loader = make_loader(val_df, False)
te_loader = make_loader(test_df, False)
VOCAB

## 3. Модель

Одна сеть с переключателями use_text и use_stars, так три варианта отличаются только входами, а голова у всех одинаковая. Текст идёт через свёртки 3, 4, 5 с max-pooling, звезда через обучаемый эмбеддинг на пять значений, признаки склеиваются, dropout и линейный слой на два класса.

In [ ]:
class MismatchNet(nn.Module):
    def __init__(self, vocab, use_text=True, use_stars=True, emb=128, n_filters=100, ks=(3, 4, 5), star_dim=16, drop=0.5):
        super().__init__()
        self.use_text = use_text
        self.use_stars = use_stars
        feat = 0
        if use_text:
            self.emb = nn.Embedding(vocab, emb, padding_idx=PAD_IDX)
            self.convs = nn.ModuleList([nn.Conv1d(emb, n_filters, k) for k in ks])
            feat += n_filters * len(ks)
        if use_stars:
            self.star_emb = nn.Embedding(5, star_dim)
            feat += star_dim
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(feat, 2)
    def forward(self, ids, star):
        parts = []
        if self.use_text:
            e = self.emb(ids).transpose(1, 2)
            parts.append(torch.cat([torch.relu(c(e)).max(dim=2).values for c in self.convs], dim=1))
        if self.use_stars:
            parts.append(self.star_emb(star))
        return self.fc(self.drop(torch.cat(parts, dim=1)))

def n_params(m):
    return sum(p.numel() for p in m.parameters())

## 4. Цикл обучения

Бинарная классификация, метрика ROC-AUC. Берём веса лучшей по val AUC эпохи.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def run_epoch(model, loader, train, opt):
    if train:
        model.train()
    else:
        model.eval()
    tot = 0
    loss_sum = 0.0
    probs = []
    gts = []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for ids, star, y in loader:
            ids = ids.to(device)
            star = star.to(device)
            y = y.to(device)
            logits = model(ids, star)
            loss = loss_fn(logits, y)
            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()
            loss_sum += loss.item() * len(y)
            tot += len(y)
            probs += torch.softmax(logits, 1)[:, 1].detach().cpu().tolist()
            gts += y.cpu().tolist()
    preds = [p > 0.5 for p in probs]
    return loss_sum / tot, accuracy_score(gts, preds), f1_score(gts, preds), roc_auc_score(gts, probs)

def train_model(name, model, epochs=EPOCHS, lr=1e-3):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    best_auc = -1.0
    best_state = None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        trl, _, _, _ = run_epoch(model, tr_loader, True, opt)
        vll, vaa, vaf, vau = run_epoch(model, va_loader, False, opt)
        hist.append({"epoch": ep, "train_loss": trl, "val_loss": vll, "val_acc": vaa, "val_f1": vaf, "val_auc": vau})
        if vau > best_auc:
            best_auc = vau
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"[{name}] epoch {ep}/{epochs} val_auc {vau} val_f1 {vaf} {time.time()-t0}s")
    if best_state:
        model.load_state_dict(best_state)
    return hist

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    probs = []
    gts = []
    for ids, star, y in loader:
        ids = ids.to(device)
        star = star.to(device)
        probs += torch.softmax(model(ids, star), 1)[:, 1].cpu().tolist()
        gts += y.tolist()
    return np.array(gts), np.array(probs)

## Логирование в MLflow

Метрики пишем в mlflow.db, веса каждого прогона в logs.

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{_PROJECT}/mlflow.db")
EXPERIMENT = "task2_mismatch"
mlflow.set_experiment(EXPERIMENT)
LOGS = _PROJECT / "logs" / EXPERIMENT

def log_dir(name):
    safe = re.sub(r"[^\w.-]+", "_", name).strip("_.") or "run"
    d = LOGS / safe
    d.mkdir(parents=True, exist_ok=True)
    return d

def log_run(name, model, hist, gts, probs):
    if not ENABLE_LOGGING:
        return
    pred = probs > 0.5
    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_param("params", n_params(model))
        mlflow.log_param("epochs", len(hist))
        mlflow.log_param("train_size", len(train_df))
        mlflow.log_param("vocab", VOCAB)
        mlflow.log_param("maxlen", MAXLEN)
        for h in hist:
            mlflow.log_metric("train_loss", h["train_loss"], step=h["epoch"])
            mlflow.log_metric("val_loss", h["val_loss"], step=h["epoch"])
            mlflow.log_metric("val_acc", h["val_acc"], step=h["epoch"])
            mlflow.log_metric("val_f1", h["val_f1"], step=h["epoch"])
            mlflow.log_metric("val_auc", h["val_auc"], step=h["epoch"])
        mlflow.log_metric("test_auc", roc_auc_score(gts, probs))
        mlflow.log_metric("test_f1", f1_score(gts, pred))
        mlflow.log_metric("test_acc", accuracy_score(gts, pred))
        torch.save(model.state_dict(), log_dir(name) / "model.pt")

## 5. Обучаем три модели

Один и тот же код, меняются только входы: звезда, текст, и то и другое.

In [ ]:
configs = [
    ("stars-only", dict(use_text=False, use_stars=True)),
    ("text-only", dict(use_text=True, use_stars=False)),
    ("text+stars", dict(use_text=True, use_stars=True)),
]
results = {}
for name, kw in configs:
    torch.manual_seed(SEED)
    model = MismatchNet(VOCAB, **kw)
    hist = train_model(name, model)
    gts, probs = evaluate(model, te_loader)
    results[name] = {"hist": hist, "gts": gts, "probs": probs, "params": n_params(model)}
    log_run(name, model, hist, gts, probs)
    print(name, "test ROC-AUC", round(roc_auc_score(gts, probs), 3))

## 6. Сравнение

In [ ]:
rows = []
for name, r in results.items():
    pred = r["probs"] > 0.5
    rows.append({"model": name, "params": r["params"], "test_auc": roc_auc_score(r["gts"], r["probs"]), "test_f1": f1_score(r["gts"], pred), "test_acc": accuracy_score(r["gts"], pred)})
comp = pd.DataFrame(rows).set_index("model").round(3)
comp

In [ ]:
plt.figure(figsize=(6, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(r["gts"], r["probs"])
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1])
plt.title("ROC-кривые")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
savefig("roc_curves")
plt.show()

## 7. Порог и подсказка автору

Метки синтетические, поэтому операционную точку выбираем честно. Смотрим, как у основной модели text+stars распределена вероятность рассогласования, и подбираем порог под сценарий, когда автору стоит переспросить, действительно ли он хотел такую оценку.

In [ ]:
gts = results["text+stars"]["gts"]
probs = results["text+stars"]["probs"]
plt.figure(figsize=(8, 4))
plt.hist(probs[gts == 0], bins=40, alpha=0.6, label="label 0")
plt.hist(probs[gts == 1], bins=40, alpha=0.6, label="label 1")
plt.title("вероятность рассогласования на тесте")
plt.xlabel("p_mismatch")
plt.legend()
savefig("prob_hist")
plt.show()

In [ ]:
thr_rows = []
for thr in [0.5, 0.7, 0.8, 0.9]:
    pred = probs > thr
    p, r, _, _ = precision_recall_fscore_support(gts, pred, average="binary", zero_division=0)
    thr_rows.append({"thr": thr, "precision": p, "recall": r, "flagged": pred.mean()})
pd.DataFrame(thr_rows).round(3)

Несколько отзывов, которые модель уверенно пометила как рассогласованные.

In [ ]:
view = test_df.reset_index(drop=True).copy()
view["p_mismatch"] = probs
flagged = view[view["p_mismatch"] > 0.9].sort_values("p_mismatch", ascending=False)
for _, row in flagged.head(4).iterrows():
    print(round(row["p_mismatch"], 2), "stars", row["stars"], "label", row["label"])
    print(str(row["text"])[:160])

## 8. Выводы

Числа в таблице выше. По одной звезде модель почти не отличает классы, AUC около 0.5, потому что распределение оценок в обоих классах мы выровняли специально. Основной сигнал несёт текст: модель смотрит, насколько он соответствует выставленной оценке, и звезда сама по себе лазейки не даёт.

В продукте это работает так. При публикации отзыва считаем вероятность рассогласования и при высоком значении ненавязчиво переспрашиваем автора. Офлайн те же отзывы понижаем в весе при расчёте рейтинга и помечаем как шумные метки для рекомендателя.

Оговорка важная. Метки синтетические, мы сами портили оценку, так что это детектор рассогласования, а не фейка. Реальные накрутки, сарказм и смешанные отзывы он поймает лишь частично, а порог подобран на синтетике. Для продакшна нужна проверка на ручной разметке.